In [0]:
# Cell 1 - Setup
storage_account_name = "stkalshianalytics"
storage_account_key = dbutils.secrets.get("kalshi-scope", "adls-storage-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

print("Ready")


In [0]:
# Cell 2 - Read Bronze and Write Silver
from pyspark.sql.functions import col, current_timestamp
from datetime import datetime, timezone

silver_start = datetime.now(timezone.utc)  # FIX: moved to top

# Dynamically find latest bronze partition
year_folders = sorted([f.name.rstrip('/') for f in dbutils.fs.ls(f"{bronze_path}/")])
latest_year = year_folders[-1]

month_folders = sorted([f.name.rstrip('/') for f in dbutils.fs.ls(f"{bronze_path}/{latest_year}/")])
latest_month = month_folders[-1]

day_folders = sorted([f.name.rstrip('/') for f in dbutils.fs.ls(f"{bronze_path}/{latest_year}/{latest_month}/")])
latest_day = day_folders[-1]

bronze_folder = f"{bronze_path}/{latest_year}/{latest_month}/{latest_day}/"
single_file = f"{bronze_path}/{latest_year}/{latest_month}/{latest_day}/page_0000.json"

print(f"Using bronze date: {latest_year}/{latest_month}/{latest_day}")

# Read schema from first page
schema = spark.read.option("multiline", "true").json(single_file).schema
df_raw = spark.read.option("multiline", "true").schema(schema).json(bronze_folder)

df_silver = df_raw.select(
    col("ticker"),
    col("title"),
    col("event_ticker"),
    col("status"),
    col("market_type"),
    col("strike_type"),
    col("no_sub_title"),
    col("yes_sub_title"),
    col("yes_ask_dollars").cast("double").alias("yes_ask"),
    col("yes_bid_dollars").cast("double").alias("yes_bid"),
    col("no_ask_dollars").cast("double").alias("no_ask"),
    col("no_bid_dollars").cast("double").alias("no_bid"),
    col("last_price_dollars").cast("double").alias("last_price"),
    col("liquidity_dollars").cast("double").alias("liquidity"),
    col("volume_24h_fp").cast("double").alias("volume_24h"),
    col("volume_fp").cast("double").alias("volume_total"),
    col("open_interest_fp").cast("double").alias("open_interest"),
    col("close_time").cast("timestamp"),
    col("open_time").cast("timestamp"),
    col("expected_expiration_time").cast("timestamp"),
    col("is_provisional"),
    col("mve_collection_ticker"),
    col("result"),
    col("can_close_early"),
    current_timestamp().alias("silver_updated_at")
).filter(
    (col("market_type") == "binary") &
    (col("mve_collection_ticker").isNull())
)

# FIX: count once and reuse
silver_count = df_silver.count()
print(f"Silver row count: {silver_count}")

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{silver_path}/kalshi_markets")
)

print("Silver Delta table written successfully")


In [0]:
# Cell 2b - Read Events and Enrich Silver
from pyspark.sql.functions import col, coalesce

# Read events dimension file from same bronze date
events_file = f"{bronze_path}/{latest_year}/{latest_month}/{latest_day}/events.json"

# Check if events file exists (wont exist for historical dates before today)
try:
    df_events = spark.read.option("multiline", "true").json(events_file).select(
        col("event_ticker"),
        col("category").alias("kalshi_category"),
        col("series_ticker"),
        col("title").alias("event_title")
    )
    events_available = True
    print(f"Events loaded: {df_events.count()} events")
except Exception as e:
    print(f"No events file found, skipping join: {e}")
    events_available = False

# Join events category to silver if available
if events_available:
    df_silver = df_silver.join(
        df_events,
        on="event_ticker",
        how="left"
    )
else:
    from pyspark.sql.functions import lit
    df_silver = df_silver \
        .withColumn("kalshi_category", lit(None).cast("string")) \
        .withColumn("series_ticker", lit(None).cast("string")) \
        .withColumn("event_title", lit(None).cast("string"))

# Overwrite silver with enriched version
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{silver_path}/kalshi_markets")
)

print("Silver enriched with events category and written successfully")

In [0]:
# Cell 3 - Metadata Logging
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

sql_server = "sql-edgar-analytics.database.windows.net"
sql_database = "edgar-analytics"
sql_user = "edgaradmin"
sql_password = dbutils.secrets.get("kalshi-scope", "sql-password")

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};user={sql_user};password={sql_password};encrypt=true;trustServerCertificate=false"

# FIX: duration now captures full silver processing time
silver_duration = (datetime.now(timezone.utc) - silver_start).total_seconds()

metadata_schema = StructType([
    StructField("layer", StringType(), False),
    StructField("source", StringType(), False),
    StructField("records_added", IntegerType(), False),
    StructField("row_count", IntegerType(), False),
    StructField("duration_sec", DoubleType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True),
    StructField("last_refresh", TimestampType(), False)
])

metadata_df = spark.createDataFrame(
    [{
        "layer": "silver",
        "source": "Databricks (02_silver_kalshi)",
        "records_added": int(silver_count),
        "row_count": int(silver_count),
        "duration_sec": round(silver_duration, 2),
        "status": "success",
        "error_message": None,
        "last_refresh": datetime.now(timezone.utc)
    }],
    schema=metadata_schema
)

(metadata_df.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "kalshi_gold.pipeline_metadata")
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .mode("append")
    .save())

print(f"Silver complete: {silver_count} rows in {silver_duration:.1f}s")
